In [27]:
from workflow.evaluate import score
from pathlib import Path
import json

In [28]:
score_info = score(Path('seed_labels.json'), Path('./output/baseline_predictions.json'))

In [29]:
error_count_lst = []
for error_class, error_info in score_info.items():
  if error_class == 'macro_mcc':
    continue
  error_counts = error_info['tp'] + error_info['fp']
  error_count_lst.append((error_counts, error_class))

error_count_lst.sort(reverse=True)
print(error_count_lst)

[(37, 'execution_failure'), (3, 'wrong_chart'), (0, 'wrong_data'), (0, 'hard_to_read')]


In [30]:
with open('./output/baseline_predictions.json', 'r') as f:
  baseline_preds = json.load(f)

with open('seed_labels.json', 'r') as f:
  labels = json.load(f)

error_dict = dict()

for pred, label in zip(baseline_preds['predictions'], labels['labels']):
  for e in pred['errors']:
    if e['family'] not in error_dict:
      error_dict[e['family']] = [(e['evidence'], pred['run_id'])]
    else:
      error_dict[e['family']].append((e['evidence'], pred['run_id']))

error_count_dict = {
  k: len(v) for k, v in error_dict.items()
}

In [33]:
label

{'run_id': '216',
 'errors': [{'family': 'wrong_data',
   'evidence': "the bar series 'Agree' holds [62.0, 71.0, 28.0, 45.0]; the request asks for [62, 71, 45, 28]"}]}

In [66]:
def get_error_details(pred_path):
  with open('seed_labels.json', 'r') as f:
    labels = json.load(f)

  with open(pred_path, 'r') as f:
    preds = json.load(f)
  
  error_details = dict()
  for pred, label in zip(preds['predictions'], labels['labels']):
    pred_error_set = set([e['family'] for e in pred['errors']])
    label_error_set = set([e['family'] for e in label['errors']])

    choices = [pred, label]
    error_types = ['FP', 'FN']
    error_sets = [pred_error_set, label_error_set]

    for i in range(len(choices)):
      choice = choices[i]
      error_type = error_types[i]
    
      for e in choice['errors']:
        if e['family'] not in error_sets[1 - i]:
          item = {
            'run_id': choice['run_id'],
            'evidence': e['evidence'],
            'truth': label_error_set,
            'prediction': pred_error_set
          }
          if e['family'] not in error_details:
            error_details[e['family']] = {
              error_type : [item]
            }
          elif error_type not in error_details[e['family']]:
            error_details[e['family']][error_type] = [item]
          else:
            error_details[e['family']][error_type].append(item)
    
  return error_details
      


In [67]:
error_details = get_error_details('./output/baseline_predictions.json')

In [69]:
counts = dict()
for e in error_details['wrong_chart']['FN']:
  prediction = list(e['prediction'])
  prediction.sort()
  prediction = tuple(prediction)
  if prediction not in counts:
    counts[prediction] = 1
  else:
    counts[prediction] += 1

In [75]:
invest = [e for e in error_details['wrong_chart']['FN'] if len(e['prediction']) == 0]

In [78]:
invest

[{'run_id': '009',
  'evidence': 'the instructions ask for x tick labels rotated by 30 degrees, but they are horizontal',
  'truth': {'hard_to_read', 'wrong_chart'},
  'prediction': set()},
 {'run_id': '013',
  'evidence': '(1) the instructions forbid jet/rainbow/hsv, but the figure uses rainbow (2) the instructions set the x limits to 0..20, but the axis runs from -5 to 25',
  'truth': {'wrong_chart'},
  'prediction': set()},
 {'run_id': '016',
  'evidence': 'the instructions ask for a legend, but the figure has none',
  'truth': {'wrong_chart', 'wrong_data'},
  'prediction': set()},
 {'run_id': '021',
  'evidence': 'axes 0 is missing xlabel',
  'truth': {'wrong_chart'},
  'prediction': set()},
 {'run_id': '025',
  'evidence': '(1) The request names a colour for each of the five regions and asks the legend to show that coding, but every wedge is drawn in the same grey and the legend is an empty box with no entries. (2) The request asks for the chart to be structured on the hierarchy o

In [60]:
error_details = get_error_details('./output/pre_iteration_seed_predictions.json')